# Load input data

In [4]:
from functions import get_experiment_data

X_df3, meta_df3, batch_df3, _ = get_experiment_data(
    design_id="df3" ,
    file_path="/DATA/WGS_study/YSL/projects/Data",
    verbose=True,
)

Design ID               : df3
Design name             : prep_genus_norm
Description             : Preprocessed HIVRC, genus-level, normalized
Aggregation             : genus
Normalize               : True
Cleanset Filtering      : True
Subset studies          : None
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table           : (936, 668)
meta_data               : (936, 11)
n_batches               : 14


# Trial investigation

In [1]:
import pandas as pd
import glob

# df3 CSV 파일 찾기
df3_p1_res = pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df3/phase1/optuna_trials_2026-04-22.csv")
print(f"전체 trial 수: {len(df3_p1_res)}")

# 필터링
df_clean = df3_p1_res[
    (df3_p1_res['permanova'] > 0.01) &
    (df3_p1_res['permanova'] < 0.1) &
    (df3_p1_res['noise_ratio'] < 0.65) &
    (df3_p1_res['n_clusters'] >= 10)
].sort_values(['noise_ratio', 'permanova'], ascending=[True, True])

print(f"의미있는 trial 수: {len(df_clean)}")
print(df_clean[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']])

전체 trial 수: 4209
의미있는 trial 수: 4
      trial_number  permanova  n_clusters  noise_ratio  cutoff
3011          3721   0.053020          34     0.563034   0.050
2791          3479   0.024847          14     0.635684   0.100
1088          1520   0.018510          43     0.638889   0.100
1951          2515   0.026668          13     0.647436   0.005


# Best trial selection (trial 2515)

In [ ]:
print(df3_p1_res[df3_p1_res['trial_number'] == 2515].T)

# Run phase 2

In [2]:
from prototype_updated_phase2 import main
main(
    experiment_name="df3",
    phase=2,
    best_trial_number=2515,
    trial_res_file_phase2="/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df3/phase1/optuna_trials_2026-04-22.csv"
)

2026-05-08 16:05:13 | 로그 저장 경로: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df3/phase2/latentgee_prototype_cutoff_df3_pid1068120_2026-05-08.log
2026-05-08 16:05:13 | LatentGEE 시작 | experiment=df3 | phase=2
2026-05-08 16:05:13 | config snapshot saved: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df3/phase2/config_used.yaml
2026-05-08 16:05:13 | python == 3.10.20
2026-05-08 16:05:13 | torch == 2.2.2
2026-05-08 16:05:13 | numpy == 1.26.4
2026-05-08 16:05:13 | scikit-learn == 1.7.2
2026-05-08 16:05:13 | optuna == 3.6.1
2026-05-08 16:05:13 | hdbscan == 0.8.41
Design ID               : df3
Design name             : prep_genus_norm
Description             : Preprocessed HIVRC, genus-level, normalized
Aggregation             : genus
Normalize               : True
Cleanset Filtering      : True
Subset studies          : None
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table         

In [5]:
# from functions import zero_filter
def zero_filter(df, meta, cutoff):
    batch="Study"
    prevalence = (df > 0).sum(axis=0) / df.shape[0]
    df = df.loc[:, prevalence > best_cutoff].copy()

    row_sums = df.sum(axis=1)
    keep_sample = row_sums > 0
    df = df.loc[keep_sample].copy()
    meta = meta.loc[keep_sample.values].reset_index(drop=True)
    
    assert len(df) == len(meta)
    assert all(df.index.astype(str) == meta["SeqID"].astype(str))
    
    return df, meta, meta[batch]

best_cutoff = 0.005
X_df3_filtered, meta_df3_filtered, batch_df3_filtered = zero_filter(X_df3, meta_df3, best_cutoff)

    

In [9]:
X_df3_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df3_filtered_cutoff0.005.csv", index=True)
meta_df3_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/meta_df3_filtered_cutoff0.005.csv", index=True)
batch_df3_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/batch_df3_filtered_cutoff0.005.csv", index=True)

In [7]:
import numpy as np

# X_corrected 로드
X_corrected_df3 = pd.read_csv(
    "/DATA/WGS_study/YSL/projects/latentgee/examples/results/df3/phase2/df3_X_corrected_trial2515_cutoff0.005_2026-04-24.csv",
    index_col=0
)

# inf, NaN 처리
X_corrected_df3_clean = X_corrected_df3.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
row_sums = X_corrected_df3_clean.sum(axis=1).replace(0, 1)
X_corrected_df3_clean = X_corrected_df3_clean.div(row_sums, axis=0)

print(f"shape: {X_corrected_df3_clean.shape}")
print(f"NaN 수: {X_corrected_df3_clean.isna().sum().sum()}")
print(f"inf 수: {np.isinf(X_corrected_df3_clean.values).sum()}")

shape: (936, 353)
NaN 수: 0
inf 수: 0


In [8]:
from functions import evaluate_batch_correction
df3_result = evaluate_batch_correction(
    X_before     = X_df3_filtered,
    X_after      = X_corrected_df3_clean,
    batch_labels = batch_df3_filtered,
    bio_labels   = meta_df3_filtered['hivstatus'],
    renormalize  = True,
    label        = "df3 — LatentGEE"
)



  df3 — LatentGEE
                        Before   After  Change
Metric                                        
PERMANOVA R² (batch) ↓  0.1835  0.0490 -0.1346
PERMANOVA R² (bio) ↑    0.0041  0.0010 -0.0031
kBET acceptance rate ↑  0.2051  0.8066  0.6015
ASW (batch) → 0        -0.1002 -0.1383 -0.0381
ASW (bio) ↑            -0.0245 -0.0185  0.0060

